# Fine-tuning a Language Model for Your Texts

This tutorial walks through the complete workflow for training a spaCy language model on your own annotated corpus. By the end you will have a model trained to tag, lemmatize, and parse your texts more accurately than a generic pre-trained model.

The tutorial uses Shakespeare's *The Winter's Tale* as the example corpus, but the workflow is the same for any language and any domain. Inline comments and notes throughout explain what to change for your specific situation.

## Before you start

**You will need:**

- A CONLL-U formatted treebank file. If you already have separate train/dev/test files you can skip Step 1. If your data is in a different format, see [Getting Training Data](../../user_guide/language_model/training_data.md).
- A Python environment with spaCy and this module installed. See the [Language Model user guide](../../user_guide/language_model/index.md) for setup.
- Time: on a modern GPU a small corpus (~300 sentences) trains in a few minutes; on CPU expect 20–40 minutes.

**If either of the following applies, read the linked guide before continuing:**

- You do not yet have annotated training data in CONLL-U format → [Getting Training Data](../../user_guide/language_model/training_data.md)
- You need a base model for your language or annotation scheme → [Choosing and Setting Up Base Models](../../user_guide/language_model/base_models.md)

**No existing data for your domain, or want to build a highly specialized model?**
See [Building a Specialized Model: Iterative Workflow](advanced_workflow.ipynb). That guide works alongside this tutorial — it walks you through getting your first training data and then directs you back here for every training and fine-tuning step.

## Prerequisites

The `lexos` package must be installed in your Python environment before running this notebook. If you haven't done this yet, run the following in a terminal from the repository root:

```bash
pip install -e .
```

This installs the package in editable mode so any changes you make to the source are reflected immediately. If you are a regular user (not modifying the source), install from PyPI once the package is published:

```bash
pip install lexos
```

After installation, run the import cell below.

In [ ]:
from pathlib import Path

from lexos.language_model import LanguageModel, split_conllu, export_to_conllu, combine_conllu

## Required models

The default configuration in this tutorial uses two source models:

- **`en_core_web_sm`** — spaCy's small English model, used as the starting point for `tok2vec` and `tagger`.
- **Bundled UD English model** (`pretrained/ud_en_ewt`) — included with this tutorial, used for `morphologizer`, `trainable_lemmatizer`, and `parser`. This model was trained on the English Web Treebank with Universal Dependencies annotation.

Run the cell below to install any missing dependencies. You only need to do this once per environment.

**Working in a language other than English?** Skip the `en_core_web_sm` download and see [Choosing and Setting Up Base Models](../../user_guide/language_model/base_models.md) for how to find or create appropriate source models for your language.

In [ ]:
import subprocess
import sys

# If you are not using English, replace "en_core_web_sm" with the spaCy model
# for your language (e.g. "de_core_news_sm" for German). See the base models user guide.
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "--quiet"], check=True)

# build, setuptools, and wheel are required for model.package() on Python 3.12+.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "build", "setuptools", "wheel"], check=True)

print("Done.")

## Configuration

The cell below sets the path variables used throughout the tutorial.

**`_tutorial_dir`** — the folder where your data, model, and output will live. By default it is derived automatically from the installed package location, which works if you are running this notebook from inside the Lexos repo after `pip install -e .`.

If you are working outside the repo — for example, running this as a standalone notebook in your own project folder — replace the auto-derivation with a direct path:

```python
_tutorial_dir = Path("C:/Users/you/my_project")  # Windows
_tutorial_dir = Path("/home/you/my_project")       # Mac / Linux
```

**Project name (`winter_tale`)** — the tutorial writes all output (assets, corpus, trained model, metrics) into a subfolder named `winter_tale`. Each cell that uses this name has a comment marking it. Rename it to something meaningful for your data — pick one name and use it consistently everywhere.

**Base model paths** — the bundled `pretrained/ud_en_ewt` model is referenced inline in the `base_model=` dict in Step 2. If you are using a different UD base model, update those three paths at that cell.

In [ ]:
import lexos.language_model as _lm

# Derive the tutorial folder from the installed package path.
# Works when running from the Lexos repo with pip install -e .
#   __init__.py → language_model/ → lexos/ → src/ → repo root
_repo_root = Path(_lm.__file__).parents[3]
_tutorial_dir = _repo_root / "doc_src" / "docs" / "tutorials" / "language_model"

# Not in the Lexos repo? Replace the two lines above with a direct path to your working folder:
# _tutorial_dir = Path("C:/Users/you/my_project")

print(f"Tutorial directory: {_tutorial_dir}")
if not _tutorial_dir.exists():
    print("WARNING: Directory not found — set _tutorial_dir to your working folder above.")

---

## Step 1: Split your data

Training requires three separate datasets:

- **Train** — what the model learns from (largest portion)
- **Dev** — used during training to check progress and decide when to stop (prevents overfitting)
- **Test** — held back entirely; used for your final accuracy measurement

**If you already have separate train/dev/test files** (e.g. downloaded directly from the Universal Dependencies project), skip this step and pass them directly to `copy_assets()` in Step 3:

```python
model.copy_assets(
    train="path/to/corpus-train.conllu",
    dev="path/to/corpus-dev.conllu",
    test="path/to/corpus-test.conllu",
)
```

**If you don't have CONLL-U data yet**, see [Getting Training Data](../../user_guide/language_model/training_data.md).

The default 80/10/10 split below is a reasonable starting point. With very small corpora (< 500 sentences) consider 90/5/5 to maximise training data.

In [ ]:
splits = split_conllu(
    input_path=str(_tutorial_dir / "wt_sanitized.conllu"),           # ← your CONLL-U data file
    output_dir=str(_tutorial_dir / "winter_tale" / "assets" / "en"), # ← rename "winter_tale"
    train_ratio=0.8,
    dev_ratio=0.1,
    seed=42,
)

for name, path in splits.items():
    print(f"{name:5s}: {path}")

**Options you can adjust:**
- `shuffle=False` — keeps sentences in document order (useful if you want Act 1 in train, Act 5 in test)
- `include_test=False` — omit the test split if you plan to evaluate separately

---

## Step 2: Create the model

Calling `LanguageModel()` creates the working directory structure and generates a training configuration. It does **not** start training yet.

The `base_model` dict controls where each pipeline component starts from. The default below uses the best available English starting point for each component — `en_core_web_sm` for components it covers well, and the bundled UD English model for the rest.

**What to change for your use case:**

| Situation | What to change |
| --- | --- |
| Different English domain (e.g. legal, medical) | Keep the defaults; your fine-tuning data will adapt the model |
| Different language | Replace all `base_model` values with models for your language — see [Choosing and Setting Up Base Models](../../user_guide/language_model/base_models.md) |
| No suitable base model exists | Omit `base_model` entirely to train from scratch, or train a base model first — see [Choosing and Setting Up Base Models](../../user_guide/language_model/base_models.md) |
| Only need POS tagging (not parsing) | Set `components=["tok2vec", "tagger"]` and use only `en_core_web_sm` entries |

For a full explanation of the component sourcing strategy and how to find or train source models, see [Choosing and Setting Up Base Models](../../user_guide/language_model/base_models.md).

In [ ]:
model = LanguageModel(
    model_dir=str(_tutorial_dir / "winter_tale"),  # ← rename "winter_tale" to your project name
    lang="en",       # ← your language BCP-47 code (e.g. "de", "fr", "es")
    gpu=False,       # ← set to True to use GPU (requires cupy — see the Language Model user guide)
    base_model={
        # tok2vec and tagger: use your language's spaCy model (e.g. "de_core_news_sm") if available
        "tok2vec":              "en_core_web_sm",
        "tagger":               "en_core_web_sm",
        # morphologizer, lemmatizer, parser: path to a UD-trained model for your language.
        # The path below points to the bundled English UD model. Replace all three if using
        # a different base model — see the base models user guide.
        "morphologizer":        str(_tutorial_dir / "pretrained" / "ud_en_ewt" / "model-best"),
        "trainable_lemmatizer": str(_tutorial_dir / "pretrained" / "ud_en_ewt" / "model-best"),
        "parser":               str(_tutorial_dir / "pretrained" / "ud_en_ewt" / "model-best"),
    },
    force=True,  # regenerate config from scratch; set to False to reconnect to an existing run
)

# After this runs, winter_tale/ contains:
#   config.cfg        (the training configuration)
#   assets/en/        (where your data files live)
#   corpus/en/        (converted data will go here)
#   training/en/      (trained model will appear here)
#   metrics/en/       (evaluation output will go here)

### Changing the starting point for any component

To use a different source for any component, edit the corresponding line in the `base_model` dict in the code cell above and re-run it. For example, to use the larger `en_core_web_lg` for richer tok2vec representations:

```python
"tok2vec": "en_core_web_lg",
"tagger":  "en_core_web_lg",
```

Or to point any of the three UD components at your own trained model:

```python
"morphologizer": "path/to/your_model/model-best",
```

Mixed configurations — where some components are sourced and others train from scratch — are supported. See the [Language Model user guide](../../user_guide/language_model/index.md) ("Fine-tuning via component sourcing") for the full explanation.

### Switching languages

This tutorial uses English, but the module works for any language with a Universal Dependencies treebank. To switch, change `lang=` to the appropriate BCP-47 code and replace the `base_model` entries with models trained for your language. See the [Language Model user guide](../../user_guide/language_model/index.md) ("Using a different language") for a step-by-step guide including where to find spaCy models and UD treebanks for other languages.

---

## Step 3: Copy and convert your data

First copy the data files into the model's `assets/` folder (this creates an archival copy), then convert them to spaCy's binary format for fast loading during training.

In [ ]:
# The ** unpacks the dict from split_conllu() — equivalent to passing
# train=..., dev=..., test=... as keyword arguments.
model.copy_assets(**splits)

In [ ]:
# Convert CONLL-U files to spaCy's binary format.
# n_sents=10 groups every 10 sentences into one training document
# (more context for the model; reduce if you run out of memory).
model.convert_assets(n_sents=10)

After `convert_assets()` runs you will see `.spacy` files in `winter_tale/corpus/en/`. These are the files the training loop actually reads.

---

## Step 4: Train

Start training. A progress table prints to the console as training runs — each row appears every 200 steps and shows the current accuracy on the dev set for each component.

**What to watch for:**
- `LOSS *` values should decrease as training progresses
- `TAG_ACC`, `POS_ACC`, `MORPH_ACC`, `LEMMA_ACC`, `DEP_UAS` should increase
- Training stops automatically when dev accuracy stops improving (early stopping)

The best checkpoint (highest dev score) is saved to `training/en/model-best`. The final checkpoint is `training/en/model-last`.

In [ ]:
# train() runs validate() first by default — it checks that assets exist,
# corpus files were produced, and the config is valid. Pass skip_validation=True
# to skip the preflight check and go straight to training.
model.train()

> **Advanced:** Training behaviour is controlled by `config.cfg` in your model directory. To change the learning rate, maximum training steps, or patience, edit the file directly and then call `model.train()` again. The key settings are in `[training.optimizer]` and `[training]`. For which settings matter and sensible ranges, see [Tuning Training Settings](../../user_guide/language_model/training_settings.md).

> **Want higher accuracy and have a GPU?** If a pre-trained transformer exists for your language and domain, fine-tuning it usually beats the CNN — see [Transformer-based Training](transformer_tutorial.ipynb).

---

## Step 5: Evaluate

Now evaluate the trained model against the held-out test set. This gives you an honest measure of accuracy on data the model never saw during training.

In [ ]:
# Evaluates model-best against the test split discovered automatically.
model.evaluate()

**Reading the results:**

| Metric | What it measures | Good range |
| --- | --- | --- |
| TAG | Penn Treebank POS accuracy | 90–95% |
| POS | Universal POS accuracy | 90–95% |
| MORPH | Morphological feature accuracy | 80–95% |
| LEMMA | Lemmatization accuracy | 85–95% |
| UAS | Dependency structure (ignoring labels) | 75–90% |
| LAS | Dependency structure + label | 65–85% |

For a small corpus like one Shakespeare play, expect scores in the lower end of these ranges. More training data is the most reliable way to improve.

Full results (including per-feature and per-relation breakdowns) are saved to `metrics/en/en.json`.

---

## Step 6: Use your model

You can load and use the trained model directly without packaging it first.

In [ ]:
import spacy

# ← rename "winter_tale" to your project name
nlp = spacy.load(str(_tutorial_dir / "winter_tale" / "training" / "en" / "model-best"))

doc = nlp("If you shall chance, Camillo, to visit Bohemia on the like occasion whereon my services are now on foot.")

for token in doc:
    print(f"{token.text:20s}  POS: {token.pos_:8s}  TAG: {token.tag_:6s}  LEMMA: {token.lemma_}")

To use your model with Lexos's tokenizer, pass the path to `make_doc()`:

```python
from lexos import tokenizer
doc = tokenizer.make_doc(text, model="../winter_tale/training/en/model-best")
```

---

## Optional: Package the model

Packaging creates a pip-installable distribution that can be shared or installed in another environment. This step is optional for personal use but useful for distribution.

In [ ]:
from pathlib import Path

# Windows MAX_PATH (260 chars) is easily exceeded by the sdist builder, which
# nests the package name several levels deep inside output_dir. Use a short
# path. C:/tmp/lm_pkg works for everyone regardless of username length.
_pkg_out = Path("C:/tmp/lm_pkg")
_pkg_out.mkdir(parents=True, exist_ok=True)

model.package(
    input_dir=str(_tutorial_dir / "winter_tale" / "training" / "en" / "model-best"),  # ← rename "winter_tale"
    output_dir=str(_pkg_out),
    name="shakespeare_sm",   # ← your model name; the language prefix is added automatically
    version="1.0.0",
    force=True,
)

# After packaging, install the .tar.gz and load by package name (not by path):
#   pip install C:/tmp/lm_pkg/en_shakespeare_sm-1.0.0/dist/en_shakespeare_sm-1.0.0.tar.gz
#   spacy.load("en_shakespeare_sm")

---

## Troubleshooting

**Reconnecting after a kernel restart:** You do not need to re-run training. Run only the imports cell, the configuration cell (`import lexos.language_model as _lm ...`), and the `LanguageModel(...)` cell with `force=False` to reconnect `model` to your existing `winter_tale/` directory. Then pick up from wherever you left off.

**Not satisfied with your results? Want a more specialized model?**
The key to better models is more and better annotated training data. The iterative bootstrap-and-correct workflow in [Building a Specialized Model: Iterative Workflow](advanced_workflow.ipynb) is how you build that data efficiently — and it applies whether you are starting from scratch or improving a model that is already trained.

**Config errors before training:** Call `debug_config()` with the path to your config file:

```python
from lexos.language_model import debug_config
debug_config(str(_tutorial_dir / "winter_tale" / "config.cfg"))
```

**Data errors before training:** Call `debug_data()`. It raises a `LexosException` if errors are found so you can catch it normally:

```python
from lexos.language_model import debug_data
debug_data(str(_tutorial_dir / "winter_tale" / "config.cfg"))
```

**LEMMA scores dropped after fine-tuning:** The trainable lemmatizer needs more examples than other components to learn reliably. With fewer than ~500 training sentences, it often performs worse than the stock model's rule-based lemmatizer. Adding more training data is the fix.

**Packaging fails on Windows (path too long):** Windows has a 260-character MAX_PATH limit. The sdist builder nests the package directory several levels deep, which can exceed this limit if `output_dir` is already deeply nested. Use a short output path such as `C:/tmp/lm_pkg` — that is what the packaging cell above does by default.

**GPU not working:** Check whether an NVIDIA GPU is visible:
```python
from lexos.language_model import _has_nvidia_gpu
print(_has_nvidia_gpu())  # True if nvidia-smi finds a GPU
```
If it prints `False`, verify your CUDA driver is installed and the GPU extras are present: `pip install .[gpu]`. See the [Language Model user guide](../../user_guide/language_model/index.md) for full setup instructions.